<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_4_nested_cv_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notes on how our model-evaluation strategy has changed
This does not introduce new information. It is a restatement and elaboration on the model-evaluation strategies that were described earlier.


## The basic Train-Test split.
When we use a Train-Test split for model evaluation, we randomly select a subset of the data (e.g., 20%) to set aside, and train our model on the remaining data. Note: When doing this alongside data transformations like standardization, we must fit the standardizer only on the training data to prevent leakage, and then apply it to the test data.

---

## Train/Test with Cross-Validation
When using cross-validation with a test set, we first do the Train-Test split. Then, we take the training data and split it again into multiple cross-validation folds. We train on most of the folds and use the remaining fold for validation, repeating this until every fold has been used as a validation set. We average the performance scores across these folds to find the best hyperparameters. Then, we train a single, brand-new model on the entire training set using those best hyperparameters, and test it on the original set-aside test set.

---

## Nested Cross-Validation
With nested cross-validation, there is no original locked-away test set. Instead, all the data is used for testing via two loops. The Inner Loop handles hyperparameter tuning: it runs cross-validation to find the best hyperparameters for a specific slice of data. This is 'nested' inside an Outer Loop, which holds out a different test fold each time to evaluate the model produced by the inner loop. To gauge overall performance, we simply average the test scores from the outer loop. This tells us how well our model-building process works on unseen data.


## Studying for a Final Exam with a Massive Problem Bank.
In this analogy:
* **Training Data** = A bank of math problems you use for practice.
* **Validation Data** = Mini-quizzes you take to see if your study strategy is working.
* **Test Data** = The actual Final Exam.
* **Hyperparameter Tuning** = Changing your study strategy (e.g., trying flashcards vs. solving by graphing vs. algebraic substitution) to see what gets you the best score on the mini-quizzes.

## Stage 1: The Simple Train / Test Split
When we first learned Machine Learning, we split our data once.

**The Analogy:** You have a bank of 1,000 math problems. You practice on 800 of them (Train), and then you take a final exam made of the remaining 200 problems (Test).

**The Problem:** Luck. What if the 200 exam problems happen to feature the exact type of fractions you struggle with? Or what if they are unusually easy?

Because we only test once, our evaluation of the model is highly dependent on how the data was randomly split. We have high variance in our evaluation.

## Stage 2: Cross-Validation + Held-Out Test Set
To fix the "lucky/unlucky split" problem, we introduced K-Fold Cross Validation. But because we still need a final, unbiased evaluation, we lock away a Test set at the very beginning.

**The Analogy:** You lock 200 problems away in a vault (Test set). With the 800 problems you have left, you divide them into 5 blocks of 160 problems. You practice on 4 blocks and take a mini-quiz on the 5th. You rotate this 5 times so every one of the 800 problems gets used for practice and for a quiz.

Crucially, this is where we do Hyperparameter Tuning. We try different study strategies (Graphing, Note Cards, etc.). We pick the strategy that gives us the highest average score across all the mini-quizzes. We then go back to the full Training set and study using the optimal strategy. Finally, to see how well we've done we take the hold-out test set that we put aside at the beginning.

**Issues with this approach:** This approach great for large datasets, but on smaller ones:
1. **Wasted Data:** We locked 20% of our precious data in a drawer. The model never got to train on it before the final evaluation!
2. **Overfitting to the Validation Set:** If you tune 1,000 different combinations of hyperparameters, you are eventually going to find a combination that performs amazingly on the mini-quizzes purely by chance.
3. **The Unlucky Test Set Returns:** Because we tuned so heavily, the locked-away Test set is our only true measure. But what if that single Test set is an unlucky split? We are right back to the Stage 1 problem.

## Stage 3: Nested Cross-Validation
How do we evaluate a heavily-tuned model without locking away 20% of our data forever? Nested cross-validation.

Instead of having a single Test set, we use cross-validation to create our Test sets (the Outer Loop), and inside of that, we use cross-validation again to tune our hyperparameters (the Inner Loop).

**The Analogy:**
Imagine a textbook with 1,000 practice problems. You divide all 1,000 problems into 5 blocks of 200.

1. **Outer Fold 1:** You lock away Block 1 to be your first Final Exam. You use the remaining 800 problems to figure out your best study strategy (via mini-quizzes/Inner Loop). Let's say you discover Flashcards work best. You study the 800 problems using Flashcards, take the Final Exam on Block 1, and score an **88%**. You write "Exam 1: 88%" on a whiteboard.
2. **The Memory Wipe:** Now, you completely wipe your memory of everything. You forget the 800 problems. You forget the 200 Final Exam problems. You even forget that Flashcards was the winning strategy.
   * *Wait, what are you retaining?* Only the score on the whiteboard. You are keeping a record of how well your study process worked, but you are throwing away the "brain" you just built.
3. **Outer Fold 2:** Now, you lock away Block 2 to be your second "Final Exam". You use the remaining 800 problems (notice that the original Block 1 is now back in the practice pile!) to figure out your best strategy. Because the practice problems are slightly different, this time you discover Group Study works best (lol). You take the Final Exam on Block 2 and score a **92%**. You write "Exam 2: 92%" on the whiteboard. You wipe your memory again.
4. You repeat this until every Block has been used as a Final Exam exactly once.

### Why is this so powerful?
Look at your whiteboard. You have 5 scores (e.g., 88%, 92%, 85%, 90%, 95%). The average is **90%**.

Because every single problem was used on a Final Exam exactly once, you haven't wasted any data, and you haven't cheated. We are no longer evaluating a single model with a specific set of hyperparameters. We are evaluating our model building process. We just mathematically proved: *"If I take a subset of data, run a grid search to tune hyperparameters, and test it on unseen data, this overall process yields a model that is about 90% accurate."*

### Wait... What is the Final Product?
This is the biggest point of confusion for students. After running Nested CV, you have 5 different models that used 5 different sets of hyperparameters. Which one do you give to your boss? Which one do you deploy to the app?

The answer is: NONE OF THEM.

The final product of Nested Cross-Validation is NOT a model. The final product is a trustworthy, unbiased number (that 90% on your whiteboard). Nested CV is strictly an evaluation tool.

Once you have used Nested CV to prove to your boss that your algorithm and tuning process works (and will yield ~90% accuracy), you are finally ready to build your production model.

**To get your Final Production Model:**
1. You take 100% of your data (all 1,000 problems).
2. You run your Inner Loop (GridSearchCV) one last time on all of it to find the absolute best hyperparameters.
3. You train one final model on 100% of the data using those winning hyperparameters.
4. You deploy that model to the real world, confident that it will perform at roughly 90% accuracy!

### Summary: When to use what?

1. **Train/Test Split:** Use for very quick and dirty baselines, or if you have an absolutely massive dataset (Millions of rows) where "lucky splits" are statistically impossible.
2. **CV + Held-out Test Set:** Use when you have a medium-to-large dataset, and you are only doing light hyperparameter tuning.
3. **Nested CV:** Use when you have a small dataset (e.g., medical data, thousands of rows or less) AND you are testing many different models or doing heavy hyperparameter tuning. It prevents data leakage and ensures your final performance metric is mathematically sound.